# jaxctrl: From Data to Optimal Control

This notebook demonstrates jaxctrl's unique **data-to-control pipeline**:

1. **SINDy** identifies a dynamical system from noisy trajectory data
2. **Controllability analysis** verifies the identified system is controllable
3. **LQR design** computes the optimal feedback gain
4. **Simulation** compares controlled vs uncontrolled trajectories
5. **Differentiable design** uses `jax.grad` to tune controller parameters

Everything is fully differentiable end-to-end.

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import jaxctrl

## 1. Generate training data

Simulate a **damped oscillator** with known dynamics:

$$\dot{x} = \begin{bmatrix} 0 & 1 \\ -1 & -0.5 \end{bmatrix} x$$

We'll pretend we don't know this matrix and recover it from data.

In [ ]:
# True system
A_true = jnp.array([[0.0, 1.0], [-1.0, -0.5]])
B = jnp.array([[0.0], [1.0]])  # control input on acceleration

# Generate trajectory via Euler integration
dt = 0.01
n_steps = 2000
key = jax.random.PRNGKey(0)

x = jnp.array([2.0, 0.0])  # initial condition
X_data = [x]

for _ in range(n_steps):
    dx = A_true @ x
    x = x + dt * dx
    X_data.append(x)

X_data = jnp.array(X_data)  # (n_steps+1, 2)

# Compute derivatives via finite differences
X = X_data[:-1]
dX = (X_data[1:] - X_data[:-1]) / dt

# Add measurement noise
noise = 0.01 * jax.random.normal(key, dX.shape)
dX_noisy = dX + noise

print(f'Training data: {X.shape[0]} samples, {X.shape[1]} states')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(X[:, 0], X[:, 1], 'b-', alpha=0.5)
ax[0].set_xlabel('x1 (position)'); ax[0].set_ylabel('x2 (velocity)')
ax[0].set_title('Phase portrait')
ax[1].plot(jnp.arange(len(X)) * dt, X)
ax[1].set_xlabel('Time'); ax[1].legend(['x1', 'x2'])
ax[1].set_title('State trajectories')
plt.tight_layout(); plt.show()

## 2. SINDy identification

Use `jaxctrl.SINDyOptimizer` with a polynomial library to discover the governing equations.

In [ ]:
# Fit SINDy model
sindy = jaxctrl.SINDyOptimizer(threshold=0.05, max_iter=20)
Xi = sindy.fit(X, dX_noisy, jaxctrl.polynomial_library)

print('SINDy coefficient matrix (rows = library terms, cols = states):')
print(Xi.round(3))

# Extract the linear system matrix
A_identified = sindy.linearize(Xi, n_vars=2)

print(f'\nTrue A:\n{A_true}')
print(f'\nIdentified A:\n{A_identified.round(4)}')
print(f'\nMax error: {jnp.max(jnp.abs(A_true - A_identified)):.4f}')

## 3. Controllability analysis

Before designing a controller, verify the identified system is controllable.

In [ ]:
print(f'Controllable: {jaxctrl.is_controllable(A_identified, B)}')
print(f'Stabilizable: {jaxctrl.is_stabilizable(A_identified, B)}')
print(f'Stable (open-loop): {jaxctrl.is_stable(A_identified)}')

# Open-loop eigenvalues
eigvals = jnp.linalg.eigvals(A_identified)
print(f'\nOpen-loop eigenvalues: {eigvals}')

## 4. LQR design

Compute the optimal state-feedback gain $u = -Kx$ that minimizes
$J = \int_0^\infty (x^T Q x + u^T R u)\, dt$.

In [ ]:
Q_cost = jnp.eye(2)
R_cost = jnp.array([[0.1]])  # cheap control

K, P = jaxctrl.lqr(A_identified, B, Q_cost, R_cost)

print(f'Optimal gain K = {K.round(4)}')

A_cl = A_identified - B @ K
cl_eigvals = jnp.linalg.eigvals(A_cl)
print(f'Closed-loop eigenvalues: {cl_eigvals.round(4)}')
print(f'All stable: {jnp.all(jnp.real(cl_eigvals) < 0)}')

## 5. Simulate: controlled vs uncontrolled

Compare the open-loop (uncontrolled) and closed-loop (LQR) responses.

In [ ]:
x0 = jnp.array([2.0, 1.0])
T = 10.0

# Open-loop (no control)
ts_ol, xs_ol = jaxctrl.simulate_lti(
    A_identified, B, x0,
    lambda t, x: jnp.zeros(1),
    T=T, num_steps=500
)

# Closed-loop (LQR)
ts_cl, xs_cl, us_cl = jaxctrl.simulate_closed_loop(
    A_identified, B, K, x0, T=T, num_steps=500
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(ts_ol, xs_ol[:, 0], 'r--', label='Open-loop')
axes[0].plot(ts_cl, xs_cl[:, 0], 'b-', label='LQR')
axes[0].set_xlabel('Time'); axes[0].set_ylabel('x1 (position)')
axes[0].legend(); axes[0].set_title('Position')

axes[1].plot(ts_ol, xs_ol[:, 1], 'r--', label='Open-loop')
axes[1].plot(ts_cl, xs_cl[:, 1], 'b-', label='LQR')
axes[1].set_xlabel('Time'); axes[1].set_ylabel('x2 (velocity)')
axes[1].legend(); axes[1].set_title('Velocity')

axes[2].plot(ts_cl, us_cl[:, 0], 'g-')
axes[2].set_xlabel('Time'); axes[2].set_ylabel('u (control input)')
axes[2].set_title('Control effort')

plt.tight_layout(); plt.show()

## 6. Differentiable design

jaxctrl's solvers support `jax.grad` through the Riccati equation.
We can compute how the LQR cost changes as we vary the state penalty $Q$.

In [ ]:
def lqr_cost(q_diag):
    """LQR cost x0^T P x0 as a function of Q = diag(q_diag)."""
    Q_ = jnp.diag(q_diag)
    _, P_ = jaxctrl.lqr(A_identified, B, Q_, R_cost)
    return x0 @ P_ @ x0

# Gradient of cost w.r.t. Q diagonal
q = jnp.array([1.0, 1.0])
cost_val = lqr_cost(q)
grad_q = jax.grad(lqr_cost)(q)

print(f'LQR cost at Q=I: {cost_val:.4f}')
print(f'dJ/dq = {grad_q.round(4)}')
print(f'  -> Increasing q1 (position penalty) changes cost by {grad_q[0]:.4f}')
print(f'  -> Increasing q2 (velocity penalty) changes cost by {grad_q[1]:.4f}')

# Sweep q1 to visualize the cost landscape
q1_range = jnp.linspace(0.1, 5.0, 50)
costs = jnp.array([lqr_cost(jnp.array([q1, 1.0])) for q1 in q1_range])
grads = jnp.array([jax.grad(lqr_cost)(jnp.array([q1, 1.0]))[0] for q1 in q1_range])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(q1_range, costs, 'b-')
ax1.set_xlabel('q1 (position penalty)'); ax1.set_ylabel('LQR cost')
ax1.set_title('Cost landscape')

ax2.plot(q1_range, grads, 'r-')
ax2.axhline(0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('q1 (position penalty)'); ax2.set_ylabel('dJ/dq1')
ax2.set_title('Cost gradient (via jax.grad)')

plt.tight_layout(); plt.show()